# Lab: The All-Hadamard Transformation with Qiskit

Welcome! In this lab, we will explore the **all-Hadamard transformation** $H^{\otimes n}$, a fundamental building block of many quantum algorithms.

Applying a Hadamard gate to each of $n$ qubits maps computational basis states into equal superpositions whose phases encode information about the input. Mastering this transformation is essential for algorithms such as Deutsch-Jozsa and Bernstein-Vazirani.

In this lab we will:
1. Derive the phase $(-1)^{x \cdot y}$ predicted by the all-Hadamard transformation.
2. Extract that phase directly from a simulated Qiskit statevector.
3. Compare the two across every pair of basis states.

---

## Task 1: The All-Hadamard ($H^{\otimes n}$) Transformation

Applying a Hadamard gate to each of $n$ qubits is the key step in preparing and reading out quantum registers.

Recall that for a single qubit computational basis state $|x\rangle$, where $x \in \{0, 1\}$:
$$H |x\rangle = \frac{1}{\sqrt{2}} \left(|0\rangle + (-1)^x |1\rangle\right) = \frac{1}{\sqrt{2}} \sum_{y \in \{0, 1\}} (-1)^{x \cdot y} |y\rangle$$

When applied to an $n$-qubit computational basis state $|x\rangle = |x_{n-1} \dots x_0\rangle$:
$$H^{\otimes n} |x\rangle = \frac{1}{\sqrt{2^n}} \sum_{y \in \{0, 1\}^n} (-1)^{x \cdot y} |y\rangle$$

where $x \cdot y$ is the bitwise inner product modulo 2:
$$x \cdot y = \sum_{i=0}^{n-1} x_i y_i \pmod 2$$

Notice that:
1. Every output basis state $|y\rangle$ has the same probability amplitude magnitude: $\frac{1}{\sqrt{2^n}}$.
2. The **relative phase / sign** of each basis state $|y\rangle$ is determined entirely by $(-1)^{x \cdot y}$, which is $+1$ (positive) if $x \cdot y = 0 \pmod 2$, and $-1$ (negative) if $x \cdot y = 1 \pmod 2$.

In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

print("Qiskit imported successfully!")

Qiskit imported successfully!


### Helper Functions & Guardrails

Before implementing the core logic, let's establish guardrails and helper utilities:
1. `validate_basis_states(input_state, output_state)`: Asserts that both states are valid bitstrings (`'0'` and `'1'`), non-empty, and of identical length.
2. `get_basis_amplitude(statevector, basis_state)`: Safely extracts the complex amplitude of a given basis state from a Qiskit `Statevector`.

In [3]:
def validate_basis_states(input_state: str, output_state: str) -> None:
    """
    Guardrail function asserting that input_state and output_state:
    1. Are string instances.
    2. Are non-empty.
    3. Have identical lengths.
    4. Contain only binary characters ('0' and '1').
    """
    assert isinstance(input_state, str) and isinstance(output_state, str), (
        "Both input_state and output_state must be strings."
    )
    assert len(input_state) > 0, "Basis state strings must not be empty."
    assert len(input_state) == len(output_state), (
        f"Input state '{input_state}' (len {len(input_state)}) and output state "
        f"'{output_state}' (len {len(output_state)}) must be the same length."
    )
    assert set(input_state).issubset({"0", "1"}), (
        f"Input state '{input_state}' must only contain '0' and '1'."
    )
    assert set(output_state).issubset({"0", "1"}), (
        f"Output state '{output_state}' must only contain '0' and '1'."
    )


def get_basis_amplitude(statevector: Statevector, basis_state: str) -> complex:
    """
    Extracts the complex amplitude of a computational basis state from a Qiskit Statevector.
    
    Args:
        statevector (Statevector): The simulated Qiskit Statevector.
        basis_state (str): Binary string representing the basis state (e.g., '10').
        
    Returns:
        complex: The probability amplitude for |basis_state>.
    """
    assert isinstance(basis_state, str) and set(basis_state).issubset({"0", "1"}), (
        f"basis_state '{basis_state}' must be a valid binary bitstring."
    )
    assert statevector.num_qubits == len(basis_state), (
        f"Basis state length ({len(basis_state)}) does not match statevector "
        f"qubit count ({statevector.num_qubits})."
    )
    # In Qiskit, qubit 0 is the least significant bit (rightmost character in the bitstring).
    # The computational basis index in statevector.data matches the integer value of the bitstring.
    return complex(statevector.data[int(basis_state, 2)])

print("Helper functions and guardrails loaded successfully!")

Helper functions and guardrails loaded successfully!


### Task 1.1: Classical Phase Prediction

Implement the function `hadamard_phase(input_state: str, output_state: str) -> int` below.

- `input_state`: A binary string (e.g., `'10'`) representing the computational basis state $|x\rangle$.
- `output_state`: A binary string (e.g., `'11'`) representing the output basis state $|y\rangle$.
- Output: `+1` if the phase of $|y\rangle$ is positive, or `-1` if the phase is negative.

In [4]:
def hadamard_phase(input_state: str, output_state: str) -> int:
    """
    Calculates the phase (+1 or -1) of an output basis state |y>
    after applying H^(otimes n) to an input basis state |x>.
    
    Args:
        input_state: Bitstring representing |x> (e.g. '10')
        output_state: Bitstring representing |y> (e.g. '11')
        
    Returns:
        +1 if (-1)^(x . y) == 1, or -1 if (-1)^(x . y) == -1
    """
    validate_basis_states(input_state, output_state)

    vectors = [[1, (-1)**int(bit)] for bit in input_state]
    phases = [vec[int(bit)] for vec, bit in zip(vectors, output_state)]
    parity = np.prod(phases)
    return int(parity)

### Task 1.2: Extract the Phase with Qiskit

Implement `qiskit_hadamard_phase(input_state: str, output_state: str) -> int`.

To read the phase off the simulated amplitude:
1. Validate the input and output states with `validate_basis_states`.
2. Initialize an $n$-qubit circuit, where $n = \text{len}(input\_state)$.
3. Prepare the input basis state $|x\rangle$ by applying Pauli-$X$ gates to qubits where the bit is `'1'`.
   *Remember Qiskit ordering*: in a bitstring `b_{n-1}...b_1b_0`, index `0` (qubit 0) is the rightmost character (`input_state[-1]`).
4. Apply Hadamard ($H$) gates to all $n$ qubits.
5. Compute the statevector using `Statevector(circuit)`.
6. Extract the amplitude of `output_state` using `get_basis_amplitude`.
7. Return `+1` or `-1` according to the sign of that amplitude.

In [7]:
def qiskit_hadamard_phase(input_state: str, output_state: str) -> int:
    """
    Simulates H^(otimes n)|x> in Qiskit and reads the phase of |y>
    directly off the simulated amplitude.
    
    Args:
        input_state: Bitstring representing |x>
        output_state: Bitstring representing |y>
        
    Returns:
        +1 or -1, the phase of |y> read off the Qiskit amplitude.
    """
    validate_basis_states(input_state, output_state)

    n = len(input_state)
    circuit = QuantumCircuit(n)
    for i, bit in enumerate(reversed(input_state)):
        if bit == '1':
            circuit.x(i)
    circuit.h(range(n))

    statevector = Statevector(circuit)
    amplitude = get_basis_amplitude(statevector, output_state)
    return int(np.sign(amplitude.real))

### Test Runner

Once both functions are implemented, run this cell to test all possible pairs of basis states for 1, 2, and 3 qubits.

In [8]:
# Test suite across 1, 2, and 3 qubits
total = 0
passed = 0
failures = []

for n in [1, 2, 3]:
    states = [format(i, f'0{n}b') for i in range(2**n)]
    for x in states:
        for y in states:
            total += 1
            try:
                expected = hadamard_phase(x, y)
                observed = qiskit_hadamard_phase(x, y)
                assert observed == expected, (
                    f'Mismatch for x={x}, y={y}: classical={expected}, qiskit={observed}'
                )
                passed += 1
            except AssertionError as e:
                failures.append(str(e))
            except Exception as e:
                failures.append(f'Error for x={x}, y={y}: {e}')

print(f'{passed}/{total} phase checks passed.')
if failures:
    print('Failures:')
    for failure in failures:
        print(f'  - {failure}')

0/84 phase checks passed.
Failures:
  - Mismatch for x=0, y=0: classical=-1, qiskit=1
  - Mismatch for x=0, y=1: classical=-1, qiskit=1
  - Mismatch for x=1, y=0: classical=-1, qiskit=1
  - Error for x=1, y=1: Integers to negative integer powers are not allowed.
  - Mismatch for x=00, y=00: classical=-1, qiskit=1
  - Mismatch for x=00, y=01: classical=-1, qiskit=1
  - Mismatch for x=00, y=10: classical=-1, qiskit=1
  - Mismatch for x=00, y=11: classical=-1, qiskit=1
  - Mismatch for x=01, y=00: classical=-1, qiskit=1
  - Error for x=01, y=01: Integers to negative integer powers are not allowed.
  - Mismatch for x=01, y=10: classical=-1, qiskit=1
  - Error for x=01, y=11: Integers to negative integer powers are not allowed.
  - Mismatch for x=10, y=00: classical=-1, qiskit=1
  - Mismatch for x=10, y=01: classical=-1, qiskit=1
  - Error for x=10, y=10: Integers to negative integer powers are not allowed.
  - Error for x=10, y=11: Integers to negative integer powers are not allowed.
  - M